In [18]:
import os
import pandas as pd
from pathlib import Path
from utils.graph_features import network_creation, features_generation
import pyarrow.parquet as pq
import pyarrow as pa

PROJECT_ROOT = Path.cwd().resolve().parent.parent
DATA_EXPLODED_PATH = PROJECT_ROOT / 'data' / 'exploded_splits'

TRAIN_PATH = DATA_EXPLODED_PATH / "train_pairs.parquet"
VAL_PATH = DATA_EXPLODED_PATH / "validation_pairs.parquet"
TEST_PATH = DATA_EXPLODED_PATH / "test_pairs.parquet"

OUTPUT_DIR =  PROJECT_ROOT / 'data' / 'graph_features'

In [10]:
# load the file 
train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
test = pd.read_parquet(TEST_PATH)

# 1. Create the Network

In [11]:
net_train = network_creation(train, "article_id", "ref_id", "is_reference_valid")
net_val = network_creation(val, "article_id", "ref_id", "is_reference_valid")
net_test = network_creation(test, "article_id", "ref_id", "is_reference_valid")

# 2. Generate the graph Features

In [12]:
final_train = features_generation(net_train, train, "article_id", "ref_id")
final_val = features_generation(net_val, val, "article_id", "ref_id")
final_test = features_generation(net_test, test, "article_id", "ref_id")

In [13]:
print("final dataset for TRAIN set:")
display(final_train.head())
print("\nfinal dataset for VALIDATION set:")
#display(final_val.head())
print("\nfinal dataset for TEST set:")
#display(final_test.head())

final dataset for TRAIN set:


,article_id,ref_id,title_article,lang_article,doc_type_article,abstract_article,year_article,n_citation_article,keywords_article,authors_article,...,in_ref,out_ref,pagerank_ref,avg_neigh_degree_ref,katz_cent_ref,degree_ratio,pagerank_ratio,pagerank_prod,common_neighbors,jaccard_coeff
0,53e99f0ab7602d97027d6a89,53e99ff0b7602d97028d14d3,Comments on a Method of Karpovsky,en,journal,Karpovsky has presented methods of multiple-va...,1978.0,12.0,"[karpovsky, method]","[{'id': '53f44aa3dabfaee43ec8ec34', 'name': 'C...",...,4,1,0.000005,0.000000,0.001216,0.500000,1.020951,2.257734e-11,1,0.142857
1,53e9bd81b7602d9704a24d06,557f4d4f6fee0fe990cb035f,The Effect of Visual Information on Word Initi...,en,conference,Disabled individuals can realize many benefits...,1996.0,5.0,"[audio-visual systems, handicapped aids, heari...","[{'id': '53f44a07dabfaeb22f4cee3a', 'name': 'R...",...,21,3,0.000014,3.333333,0.003025,0.250000,0.058128,1.182040e-11,0,0.000000
2,539087fe20f70186a0d75db6,539087ae20f70186a0d4cf5a,Recursive Cube of Rings: A New Topology for In...,en,journal,"In this paper, we introduce a family of scalab...",2000.0,21.0,"[scalable computer systems, recursive cube of ...","[{'id': '542ad493dabfae646d58b3c7', 'name': 'Y...",...,4,2,0.000002,4.000000,0.000925,1.333333,0.503580,1.363682e-12,0,0.000000
3,539087fe20f70186a0d75db6,5390878e20f70186a0d3a260,Recursive Cube of Rings: A New Topology for In...,en,journal,"In this paper, we introduce a family of scalab...",2000.0,21.0,"[scalable computer systems, recursive cube of ...","[{'id': '542ad493dabfae646d58b3c7', 'name': 'Y...",...,159,5,0.000079,2.800000,0.014876,0.666667,0.010539,6.519889e-11,0,0.000000
4,539087fe20f70186a0d75db6,539087cb20f70186a0d58fe1,Recursive Cube of Rings: A New Topology for In...,en,journal,"In this paper, we introduce a family of scalab...",2000.0,21.0,"[scalable computer systems, recursive cube of ...","[{'id': '542ad493dabfae646d58b3c7', 'name': 'Y...",...,148,4,0.000066,3.000000,0.018099,0.800000,0.012466,5.511966e-11,0,0.000000



final dataset for VALIDATION set:

final dataset for TEST set:


In [ ]:
cols_keep = ['article_id', 'ref_id']
drop_cols = ["year_article", "n_citation_article", "year_ref"]
num_cols = final_train.select_dtypes(include='number').columns

train_final = final_train[cols_keep + list(num_cols)].drop(columns=drop_cols, errors="ignore")
val_final   = final_val[cols_keep + list(num_cols)].drop(columns=drop_cols, errors="ignore")
test_final  = final_test[cols_keep + list(num_cols)].drop(columns=drop_cols, errors="ignore")

In [15]:
train_final

,article_id,ref_id,year_article,n_citation_article,year_ref,is_reference_valid,in_article,out_article,pagerank_article,avg_neigh_degree_article,...,in_ref,out_ref,pagerank_ref,avg_neigh_degree_ref,katz_cent_ref,degree_ratio,pagerank_ratio,pagerank_prod,common_neighbors,jaccard_coeff
0,53e99f0ab7602d97027d6a89,53e99ff0b7602d97028d14d3,1978.0,12.0,1977.0,1,2,1,4.801589e-06,1.000000,...,4,1,4.702055e-06,0.000000,0.001216,0.500000,1.020951,2.257734e-11,1,0.142857
1,53e9bd81b7602d9704a24d06,557f4d4f6fee0fe990cb035f,1996.0,5.0,1993.0,1,0,1,8.289391e-07,3.000000,...,21,3,1.425967e-05,3.333333,0.003025,0.250000,0.058128,1.182040e-11,0,0.000000
2,539087fe20f70186a0d75db6,539087ae20f70186a0d4cf5a,2000.0,21.0,1992.0,1,0,4,8.289391e-07,3.250000,...,4,2,1.645093e-06,4.000000,0.000925,1.333333,0.503580,1.363682e-12,0,0.000000
3,539087fe20f70186a0d75db6,5390878e20f70186a0d3a260,2000.0,21.0,1989.0,1,0,4,8.289391e-07,3.250000,...,159,5,7.865342e-05,2.800000,0.014876,0.666667,0.010539,6.519889e-11,0,0.000000
4,539087fe20f70186a0d75db6,539087cb20f70186a0d58fe1,2000.0,21.0,1990.0,1,0,4,8.289391e-07,3.250000,...,148,4,6.649423e-05,3.000000,0.018099,0.800000,0.012466,5.511966e-11,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2162515,53e9bde2b7602d9704a91def,53908aac20f70186a0da7f23,2002.0,21.0,1995.0,0,0,12,8.289391e-07,4.333333,...,0,0,8.289391e-07,0.000000,0.000652,12.000000,0.998795,6.871400e-13,0,0.000000
2162516,53e9b5b6b7602d97040fba6c,5390878320f70186a0d33894,2003.0,9.0,1988.0,0,0,2,8.289391e-07,2.000000,...,0,0,8.289391e-07,0.000000,0.000652,2.000000,0.998795,6.871400e-13,0,0.000000
2162517,53e9b5b6b7602d97040fba6c,53908b1820f70186a0db4f04,2003.0,9.0,2000.0,0,0,2,8.289391e-07,2.000000,...,0,0,8.289391e-07,0.000000,0.000652,2.000000,0.998795,6.871400e-13,0,0.000000
2162518,53e9a146b7602d9702a39e8d,53e9a539b7602d9702e59e18,2002.0,27.0,2001.0,0,0,2,8.289391e-07,1.500000,...,1,0,1.102415e-06,0.000000,0.000717,2.000000,0.751249,9.138346e-13,0,0.000000


## 3. Export features

In [19]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_final.to_parquet(OUTPUT_DIR / "final_train.parquet", index=False)
val_final.to_parquet(OUTPUT_DIR / "final_val.parquet", index=False)
test_final.to_parquet(OUTPUT_DIR / "final_test.parquet", index=False)